# 공개 레퍼런스 비교 보고서: M1 · M7 · MCF · HCF

COMSOL 공식 조건·논문을 기존 Python FEM으로 재현한 보고서입니다. COMSOL 프로그램 자체를 실행하지 않았습니다.

앞부분은 GitHub Actions에서 검증한 저장 결과를 읽습니다. 뒷부분의 선택 실행 셀은 동일한 소스 해시를 확인하고 FEM을 다시 계산합니다. Google Colab에서의 이 노트북 전체 실행은 별도로 확인하지 않았습니다. 수치 해석기는 연결된 GitHub Actions 실행에서 검증했습니다.


In [ ]:
import json, urllib.request
from IPython.display import display, Markdown, SVG
RESULTS_COMMIT = "21cb50ff48859c60932be7b3c12dc60c26832884"
BASE = f"https://raw.githubusercontent.com/kimheeseo/LSCNS/{RESULTS_COMMIT}/COMSOL/public_reference_benchmarks/results"
with urllib.request.urlopen(BASE + "/comparison.json") as response:
    saved = json.load(response)
m1 = [r for r in saved["M1"]["mesh"] if r["mode"] == "TE0"][-1]
m7 = saved["M7"]["mesh"][-1]
mcf = saved["MCF"]["extended_domain"][-1]
hcf = saved["HCF"]["mesh"][-1]
rows = [
    ("M1 TE0", f'{m1["neff"]:.12f}', f'{m1["abs_error"]:.6g}', "절대 neff / 공개 해석식"),
    ("M7 LP01", f'{m7["neff"]:.12f}', f'{m7["scalar_abs_error"]:.6g}', "절대 neff / 스칼라 기준"),
    ("MCF κ", f'{mcf["kappa_per_m"]:.9f} /m', f'{mcf["kappa_error_pct"]:.6g}%', "독립 스칼라 기준"),
    ("HCF neff", f'{hcf["neff"]:.12f}', f'{hcf["neff_abs_error"]:.6g}', "절대 neff / 모세관 근사"),
]
text = "| 대상 | FEM 값 | 오차 | 기준 |\n|---|---|---|---|\n"
text += "\n".join("| " + " | ".join(row) + " |" for row in rows)
passed = sum(c["passed"] for c in saved["checks"])
display(Markdown(f'**수치 검사 {passed}/{len(saved["checks"])} 통과**\n\n' + text))
display(Markdown("실행 기록: " + saved["metadata"]["run_url"]))
display(Markdown("**이 오차를 실제 COMSOL 또는 제품 오차로 읽으면 안 됩니다.** M7은 공개 neff=1.4444의 반올림 수준 비교, MCF는 논문 그림의 근사 비교, HCF는 모세관 식 검증이며 ARF/NANF 누설 검증은 아닙니다."))
if "published_curve_comparison" in saved["MCF"]:
    display(saved["MCF"]["published_curve_comparison"])



In [ ]:
for name in ["m1_fields.svg", "m7_convergence.svg", "mcf_gap.svg", "hcf_radius.svg"]:
    display(SVG(url=BASE + "/" + name))



## 선택: 실제 FEM 재계산

이 아래 셀은 공개 패키지와 Linux 공유 라이브러리를 설치하고, 검증한 소스 커밋의 필요한 파일만 내려받습니다. 원본 저장소 전체를 복제하지 않습니다. 계산에 수 분이 걸릴 수 있습니다. 파일 해시는 저장된 검증 기록과 대조합니다.


In [ ]:
from pathlib import Path
import hashlib, os, shutil, subprocess, sys
task_root = Path("/content/LS_public_reference") if Path("/content").exists() else Path.cwd() / "LS_public_reference"
comsol_root = task_root / "COMSOL"
source_commit = saved["metadata"]["source_commit"]
expected = saved["metadata"]["source_sha256"]
paths = list(expected) + ["waveoptics_m1/waveoptics/__init__.py", "research/requirements.txt"]
for relative in paths:
    url = f"https://raw.githubusercontent.com/kimheeseo/LSCNS/{source_commit}/COMSOL/{relative}"
    with urllib.request.urlopen(url) as response:
        body = response.read()
    if relative in expected:
        assert hashlib.sha256(body).hexdigest() == expected[relative], relative
    target = comsol_root / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(body)
(task_root / "download_manifest.json").write_text(json.dumps({"source_commit": source_commit, "verified_sha256": expected}, indent=2))
if sys.platform.startswith("linux") and hasattr(os, "geteuid") and os.geteuid() == 0:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "libglu1-mesa", "libxft2", "libxrender1", "libxinerama1", "libxcursor1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(comsol_root / "research/requirements.txt")], check=True)
subprocess.run([sys.executable, str(comsol_root / "public_reference_benchmarks/run_comparison.py")], cwd=task_root, check=True)
print("Pinned source:", source_commit)
print("The downloaded sources are not a git checkout; their provenance is in download_manifest.json.")



In [ ]:
results = comsol_root / "public_reference_benchmarks/results"
display(Markdown((results / "COMPARISON_REPORT.md").read_text()))
for filename in results.glob("*.svg"):
    display(SVG(filename=str(filename)))



In [ ]:
import zipfile
archive = task_root / "Public_Reference_Recalculated_Results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for path in results.iterdir():
        if path.is_file():
            z.write(path, "results/" + path.name)
    z.write(task_root / "download_manifest.json", "download_manifest.json")
print(archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print("Open the ZIP at the path above.")

